In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import json
import unicodedata
import plotly.graph_objects as go
import dash
from dash import dcc, html, Input, Output

/Users/santiagoquintana/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
Data = pd.read_csv('DataAWS.csv')

cols_used = ['fami_educacionmadre', 'fami_educacionpadre', #heatmap
             'cole_area_ubicacion', 'cole_caracter','cole_naturaleza','cole_jornada', #barplots apilados y donas y geograficos
             'cole_mcpio_ubicacion', 'cole_nombre_establecimiento',
             
             'punt_matematicas','punt_c_naturales'] # histograma kde
Data_used = Data[cols_used]
Data_used['punt_prom_mcn'] = (Data_used['punt_matematicas'] + Data_used['punt_c_naturales'])/2

/var/folders/_2/mzpdbswd3cq_m_g4ggyf09700000gn/T/ipykernel_53167/1164669375.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Data_used['punt_prom_mcn'] = (Data_used['punt_matematicas'] + Data_used['punt_c_naturales'])/2


In [3]:
mapeo_educativo = {'Educación profesional completa': 4,
 'Educación profesional incompleta': 3.5,
 'Ninguno': 0,
 'No Aplica': 0,
 'No sabe': 0,
 'Postgrado': 5,
 'Primaria completa': 1,
 'Primaria incompleta': 0.5,
 'Secundaria (Bachillerato) completa': 2,
 'Secundaria (Bachillerato) incompleta': 1.5,
 'Técnica o tecnológica completa': 3,
 'Técnica o tecnológica incompleta': 2.5
 }
# proemdio padre - madre mapeado

In [47]:
import plotly.graph_objects as go
import scipy.stats as stats
import numpy as np

MUNICIPIO = 'TUNJA'
VARIABLE = 'punt_prom_mcn'

data_mcpio = Data_used[Data_used['cole_mcpio_ubicacion'] == MUNICIPIO][VARIABLE].dropna()
data_boyaca = Data_used[VARIABLE].dropna()

x_range = np.linspace(0, 100, 500)
kde_boyaca = stats.gaussian_kde(data_boyaca)
y_boyaca = kde_boyaca(x_range)
y_boyaca_scaled = y_boyaca * len(data_mcpio) * (100/30) 
fig = go.Figure()


fig.add_trace(go.Histogram(
    x=data_mcpio,
    name=f'Frecuencia en {MUNICIPIO}',
    marker_color='#636EFA',
    opacity=0.6,
    nbinsx=30
))

fig.add_trace(go.Scatter(
    x=x_range, 
    y=y_boyaca_scaled,
    mode='lines',
    name='Tendencia Boyacá (KDE)',
    line=dict(color='red', width=3)
))


fig.update_layout(
    title=f"Desempeño en {VARIABLE}: {MUNICIPIO} frente al promedio de Boyacá",
    xaxis_title="Puntaje",
    yaxis_title="Cantidad de Estudiantes",
    template="plotly_white",
    bargap=0.05
)


fig.show(renderer="browser")

### MAPA

In [9]:
with open('gadm41_COL_2.json', 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)
    
with open('gadm41_COL_1.json', 'r', encoding='utf-8') as f:
    geojson_dpto = json.load(f)

def normalizar(name):
    name = name.lower().strip()
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if not unicodedata.combining(c))
    name = name.replace(' ', '')
    name = name.replace('cienega', 'cienaga')
    name = name.replace('guicandelasierra', 'guican')
    name = name.replace('pisva', 'pisba')
    name = name.replace('tutasa', 'tutaza')
    return name

boyaca_dpto = next(f for f in geojson_dpto['features'] if normalizar(f['properties']['NAME_1']) == 'boyaca')
            
todos_municipios = Data_used['cole_mcpio_ubicacion'].dropna().unique().tolist()

mapa_norm_a_real = {}
for m in todos_municipios:
    norm = normalizar(m)
    if norm not in mapa_norm_a_real:
        mapa_norm_a_real[norm] = m
    else:
        existente = mapa_norm_a_real[norm]
        count_nuevo    = len(Data_used[Data_used['cole_mcpio_ubicacion'] == m])
        count_existente = len(Data_used[Data_used['cole_mcpio_ubicacion'] == existente])
        if count_nuevo > count_existente:
            mapa_norm_a_real[norm] = m


geos_coinc = []
for feature in geojson_data['features'][212:335]:
    nombre_municipio = feature['properties']['NAME_2']
    if normalizar(nombre_municipio) in mapa_norm_a_real:
        geos_coinc.append(feature)

filt_geojson = {"type": "FeatureCollection", "features": geos_coinc}

for i, feature in enumerate(filt_geojson['features']):
    feature['id'] = i

plot_df = pd.DataFrame([
    {   "id": i,
        "name": f['properties']['NAME_2'],
        "nombre_real": mapa_norm_a_real.get(normalizar(f['properties']['NAME_2']))} for i, f in enumerate(filt_geojson['features'])
])


coords_centr = geos_coinc[0]['geometry']['coordinates'][0][0][0]
latids_bordes, longs_bordes = [], []
geom = boyaca_dpto['geometry']
polys = geom['coordinates'] if geom['type'] == 'Polygon' else geom['coordinates']
for poly in polys:
    ring = poly[0]
    longs_bordes.extend([p[0] for p in ring] + [None])
    latids_bordes.extend([p[1] for p in ring] + [None])

In [13]:
mapa_norm_a_real = {}
for m in sorted(Data_used['cole_mcpio_ubicacion'].dropna().unique()):
    norm = normalizar(m)
    if norm not in mapa_norm_a_real:
        mapa_norm_a_real[norm] = m

# Añadir nombre canónico a plot_df UNA SOLA VEZ
plot_df['nombre_real'] = plot_df['name'].apply(
    lambda x: mapa_norm_a_real.get(normalizar(x))
)


In [ ]:
vars_cat = {
    'Sin filtro': None,
    'Área de Ubicación': 'cole_area_ubicacion',
    'Naturaleza': 'cole_naturaleza',
    'Jornada': 'cole_jornada',
    'Carácter': 'cole_caracter'
}

puntajes = {
    'Matemáticas': 'punt_matematicas',
    'Ciencias Naturales': 'punt_c_naturales',
    'Promedio Mates y Cienc. Nat.': 'punt_prom_mcn'
}


def format_hover(r):
    nombre      = r['cole_mcpio_ubicacion'] if pd.notna(r.get('cole_mcpio_ubicacion')) else r['name']
    promedio    = f"{r['promedio']:.1f}" if pd.notna(r.get('promedio')) else 'N/D'
    colegios    = int(r['num_colegios']) if pd.notna(r.get('num_colegios')) else 'N/D'
    estudiantes = int(r['num_estudiantes']) if pd.notna(r.get('num_estudiantes')) else 'N/D'
    return (f"<b>{nombre}</b><br>"
            f"Puntaje promedio: {promedio}<br>"
            f"N° Colegios: {colegios}<br>"
            f"N° Estudiantes: {estudiantes}")

def agrup_municp(df, col_puntaje, col_cat=None, cat_valor=None):
    if col_cat and cat_valor:
        df = df[df[col_cat] == cat_valor]
    agg = df.groupby('cole_mcpio_ubicacion').agg(
        promedio = (col_puntaje, 'mean'),
        num_colegios = ('cole_nombre_establecimiento', 'nunique'),
        num_estudiantes = (col_puntaje, 'count')
    ).reset_index()
    return agg

def agrup_municp(df, col_puntaje, col_cat=None, cat_valor=None):
    df = df.copy()
    df['cole_mcpio_ubicacion'] = df['cole_mcpio_ubicacion'].map(
        lambda x: mapa_norm_a_real.get(normalizar(x), x)
    )
    if col_cat and cat_valor:
        df = df[df[col_cat] == cat_valor]
    agg = df.groupby('cole_mcpio_ubicacion').agg(
        promedio = (col_puntaje, 'mean'),
        num_colegios = ('cole_nombre_establecimiento', 'nunique'),
        num_estudiantes = (col_puntaje, 'count')
    ).reset_index()
    agg['mcpio_norm'] = agg['cole_mcpio_ubicacion'].apply(normalizar)
    return agg

def datos_trazar(punt_col, col_cat=None, cat_valor=None):
    agg = agrup_municp(Data_used, punt_col, col_cat, cat_valor)
    
    munic_df = plot_df.copy()
    munic_df['name_norm'] = munic_df['name'].apply(normalizar)
    munic_df = munic_df.merge(agg, left_on='name_norm', right_on='mcpio_norm', how='left')

    def hover(r):
        nombre = r['nombre_real'] if pd.notna(r.get('nombre_real')) else r['name']
        promedio = f"{r['promedio']:.1f}" if pd.notna(r.get('promedio')) else 'N/D'
        colegios = int(r['num_colegios']) if pd.notna(r.get('num_colegios')) else 'N/D'
        estudiantes = int(r['num_estudiantes']) if pd.notna(r.get('num_estudiantes')) else 'N/D'
        return (f"<b>{nombre}</b><br>"
                f"Puntaje promedio: {promedio}<br>"
                f"N° Colegios: {colegios}<br>"
                f"N° Estudiantes: {estudiantes}")

    munic_df['hover'] = munic_df.apply(hover, axis=1)
    return munic_df['promedio'].tolist(), munic_df['hover'].tolist(), plot_df['nombre_real'].tolist()

# ── App ────────────────────────────────────────────────────────────────────────
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H3("Desempeño Académico por municipio — Boyacá",
            style={'textAlign': 'center', 'fontFamily': 'Arial'}),

    # dropdowns
    html.Div([
        html.Div([
            html.Label("Variable:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='dd-cat',
                options=[{'label': k, 'value': k} for k in vars_cat.keys()],
                value='Sin filtro',
                clearable=False,
                style={'width': '250px'}
            )
        ], style={'marginRight': '20px'}),

        html.Div([
            html.Label("Valor:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='dd-val',
                options=[{'label': 'Todos', 'value': 'Todos'}],
                value='Todos',
                clearable=False,
                style={'width': '220px'}
            )
        ], style={'marginRight': '20px'}),
        
        html.Div([
            html.Label("Puntaje:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='dd-punt',
                options=[{'label': k, 'value': k} for k in puntajes.keys()],
                value='Promedio Mates y Cienc. Nat.',
                clearable=False,
                style={'width': '220px'}
            )
        ])
    ], style={'display': 'flex', 'alignItems': 'flex-end',
              'padding': '10px 20px', 'fontFamily': 'Arial'}),

    # mapa    
    dcc.Graph(id='mapa', style={'height': '80vh'}),
    
    # guardar muncipio clickeado
    dcc.Store(id='municipio-store'),

    #overlay por municipio
    html.Div(id='modal-overlay', children=[
        html.Div([
            # Header modal
            html.Div([
                html.H4(id='modal-title',
                        style={'margin': '0', 'fontFamily': 'Arial',
                               'fontSize': '15px', 'color': '#2c3e50'}),
                html.Button('✕', id='modal-close',
                            style={'background': 'none', 'border': 'none',
                                   'fontSize': '20px', 'cursor': 'pointer',
                                   'color': '#666', 'padding': '0 5px'})
            ], style={'display': 'flex', 'justifyContent': 'space-between',
                      'alignItems': 'center', 'marginBottom': '12px',
                      'borderBottom': '1px solid #eee', 'paddingBottom': '10px'}),

            # Dropdowns del modal
            html.Div([
                html.Div([
                    html.Label("Variable del colegio:",
                               style={'fontSize': '12px', 'fontWeight': 'bold',
                                      'fontFamily': 'Arial'}),
                    dcc.Dropdown(
                        id='modal-dd-cat',
                        options=[{'label': v, 'value': k}
                                 for k, v in {
                                     'cole_jornada': 'Jornada',
                                     'cole_area_ubicacion': 'Zona de Ubicación',
                                     'cole_naturaleza': 'Naturaleza',
                                     'cole_caracter': 'Carácter'
                                 }.items()],
                        value='cole_jornada',
                        clearable=False,
                        style={'width': '200px', 'fontSize': '12px'}
                    )
                ], style={'marginRight': '15px'}),
                html.Div([
                    html.Label("Puntaje:",
                               style={'fontSize': '12px', 'fontWeight': 'bold',
                                      'fontFamily': 'Arial'}),
                    dcc.Dropdown(
                        id='modal-dd-punt',
                        options=[{'label': k, 'value': v}
                                 for k, v in puntajes.items()],
                        value='punt_prom_mcn',
                        clearable=False,
                        style={'width': '220px', 'fontSize': '12px'}
                    )
                ])
            ], style={'display': 'flex', 'marginBottom': '10px'}),

            # Violin plot
            dcc.Graph(id='violin-plot',
                      style={'height': '380px'},
                      config={'displayModeBar': False}),

            # html.P("💡 Haz click en otro municipio para actualizar",
            #        style={'textAlign': 'center', 'fontSize': '11px',
            #               'color': '#aaa', 'fontFamily': 'Arial', 'margin': '5px 0 0 0'})

        ], style={
            'background': 'white',
            'borderRadius': '12px',
            'padding': '20px',
            'width': '620px',
            'boxShadow': '0 8px 32px rgba(0,0,0,0.18)',
            'position': 'relative'
        })
    ], style={
        'display': 'none',
        'position': 'fixed',
        'top': '0', 'left': '0',
        'width': '100%', 'height': '100%',
        'backgroundColor': 'rgba(0,0,0,0.45)',
        'zIndex': '1000',
        'justifyContent': 'center',
        'alignItems': 'center'
    })
])

# ── Callback 1: actualizar opciones de valor según categoría ───────────────────
@app.callback(
    Output('dd-val', 'options'),
    Output('dd-val', 'value'),
    Input('dd-cat', 'value')
)
def update_valores(cat_label):
    col_cat = vars_cat.get(cat_label)
    if not col_cat:
        return [{'label': 'Todos', 'value': 'Todos'}], 'Todos'
    valores = ['Todos'] + sorted(Data_used[col_cat].dropna().unique().tolist())
    return [{'label': v, 'value': v} for v in valores], 'Todos'

# ── Callback 2: actualizar mapa ────────────────────────────────────────────────
@app.callback(
    Output('mapa', 'figure'),
    Input('dd-cat', 'value'),
    Input('dd-val', 'value'),
    Input('dd-punt', 'value')
)
def update_mapa(cat_label, cat_val, punt_label):
    col_cat = vars_cat.get(cat_label)
    punt_col = puntajes[punt_label]
    cat_filtro = cat_val if cat_val != 'Todos' else None

    z, text, municipios_reales = datos_trazar(punt_col, col_cat=col_cat, cat_valor=cat_filtro)

    titulo = punt_label
    if col_cat:
        titulo += f" | {cat_label}" + (f": {cat_val}" if cat_val != 'Todos' else '')

    fig = go.Figure()
    fig.add_trace(go.Choroplethmap(
        geojson = filt_geojson,
        locations = plot_df['id'],
        z = z,
        customdata= municipios_reales,
        colorscale = "RdYlGn",
        zmin = Data_used[punt_col].quantile(0.05),
        zmax = Data_used[punt_col].quantile(0.95),
        marker_opacity = 0.7,
        marker_line_width= 0.5,
        text = text,
        hovertemplate = "%{text}<extra></extra>",
        colorbar = dict(title="Puntaje promedio")
    ))
    fig.add_trace(go.Scattermap(
        lat=latids_bordes, lon=longs_bordes,
        mode='lines',
        line=dict(width=1.75, color='black'),
        hoverinfo='skip', showlegend=False
    ))
    fig.update_layout(
        map_style = "carto-positron",
        map_zoom = 7,
        map_center = {"lat": coords_centr[1], "lon": coords_centr[0]},
        margin = {"r":0, "t":40, "l":0, "b":0},
        title = dict(text=f"Desempeño Puntajes Matemáticas y Ciencias Naturales — {titulo}", x=0.5, font=dict(size=15))
    )
    return fig


# ── Callback 3: guardar municipio ────────────────────────────────────────────────

@app.callback(
    Output('municipio-store', 'data'),
    Input('mapa', 'clickData'),
    prevent_initial_call=True
)
def guardar_municipio(clickData):
    if clickData is None:
        return None
    punto = clickData['points'][0]
    municipio_real = punto.get('customdata')
    print(f"DEBUG — customdata: '{municipio_real}'")
    return municipio_real

# ── Callback 4: mostrar/ocultar modal ───────────────────────────────────────────
@app.callback(
    Output('modal-overlay', 'style'),
    Input('municipio-store', 'data'),
    Input('modal-close', 'n_clicks'),
    prevent_initial_call=True
)
def municipio_mod_click(municipio, close_clicks):
    click = dash.callback_context.triggered[0]['prop_id']
    base_style = {
        'position': 'fixed', 'top': '0', 'left': '0',
        'width': '100%', 'height': '100%',
        'backgroundColor': 'rgba(0,0,0,0.45)',
        'zIndex': '1000', 'justifyContent': 'center', 'alignItems': 'center'
    }
    if 'modal-close' in click or municipio is None:
        return {**base_style, 'display': 'none'}
    return {**base_style, 'display': 'flex'}

# ── Callback: título del modal ─────────────────────────────────────────────────
@app.callback(
    Output('modal-title', 'children'),
    Input('municipio-store', 'data'),
    prevent_initial_call=True
)
def act_titulo_modal(municipio):
    if not municipio:
        return ''
    return f"{municipio.title()} — Análisis de Desempeño"


# ── Callback 3: crear visualizacion por municipio ────────────────────────────────────────────────

@app.callback(
    Output('violin-plot', 'figure'),
    Input('municipio-store', 'data'),
    Input('modal-dd-cat', 'value'),
    Input('modal-dd-punt', 'value'),
    prevent_initial_call=True
)

def update_violin(municipio, col_cat, punt_col):
    map_cole_vars = {
        'cole_area_ubicacion': 'Zona de Ubicación',
        'cole_naturaleza': 'Naturaleza',
        'cole_caracter': 'Carácter',
        'cole_jornada': 'Jornada'
    }
    map_puntajes = {
        'punt_matematicas': 'Matemáticas',
        'punt_c_naturales': 'Ciencias Naturales',
        'punt_prom_mcn': 'Promedio Mats. Ciencias Nat.'
    }
    
    if not municipio:
        return go.Figure()

    municipio_canonico = mapa_norm_a_real.get(normalizar(municipio), municipio)
    df_mun = Data_used.copy()
    df_mun['mcpio_canon'] = df_mun['cole_mcpio_ubicacion'].map(
        lambda x: mapa_norm_a_real.get(normalizar(x), x)
    )
    df_mun = df_mun[df_mun['mcpio_canon'] == municipio_canonico].dropna(subset=[col_cat, punt_col])

    if df_mun.empty:
        return go.Figure()
        
    categorias = sorted(df_mun[col_cat].unique())
    fig = go.Figure()

    for cat in categorias:
        df_cat = df_mun[df_mun[col_cat] == cat]
        proporcion = len(df_cat) / len(df_mun) * 100
        mean_val   = df_cat[punt_col].mean()

        fig.add_trace(go.Violin(
            y = df_cat[punt_col],
            name = f"{cat}<br>{proporcion:.1f}%",
            box_visible = True,
            meanline_visible = True,
            points = False,
            hoveron = None,
            hovertemplate = (
                f"<b>{cat}</b><br>"
                f"Promedio: {mean_val:.1f}<br>"
                f"Proporción: {proporcion:.1f}%<br>"
                f"N°: {len(df_cat)}<extra></extra>"
            )
        ))

    fig.update_layout(
        showlegend = False,
        yaxis_title = map_puntajes.get(punt_col, punt_col),
        xaxis_title = map_cole_vars.get(col_cat, col_cat),
        margin = {"r":20, "t":10, "l":50, "b":60},
        plot_bgcolor = 'white',
        yaxis = dict(gridcolor='#f0f0f0'),
        font = dict(family='Arial', size=11),
        hovermode = 'closest',
    )
    return fig


app.run(debug=True, port=8050)

DEBUG — customdata: 'SOGAMOSO'
DEBUG — customdata: 'TASCO'


,id,name,nombre_real
0,0,Almeida,ALMEIDA
1,1,Aquitania,AQUITANIA
2,2,Arcabuco,ARCABUCO
3,3,Belén,BELEN
4,4,Berbeo,BERBEO
...,...,...,...
118,118,Umbita,UMBITA
119,119,Ventaquemada,VENTAQUEMADA
120,120,VilladeLeyva,VILLA DE LEYVA
121,121,Viracachá,VIRACACHA
